# SIGMOD Exp2 WC=2 History 0% vs 1% Breakdown

Frozen trace comparison for the zero-point anomaly.


In [ ]:
from pathlib import Path
import sys
import random
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    display_name,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_wc2_history_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}

CONFIG = {
    'warehouse_count': 2,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.01,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'repeat': 5,
    'trim': 1,
    'timeout_sec': 900,
    'trace_seed': 232323223,
    'force_rerun': True,
}

TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}
TX_ORDER = ['InitLoad', 'Update', 'GbgCollect', 'BuildSnap', 'RecentScan', 'HistoryScan', 'Probe', 'DeltaScan']
WRITE_OPS = {'InitLoad', 'Update', 'GbgCollect', 'BuildSnap'}
COLOR_MAP = {
    'InitLoad': TOL['grey'],
    'Update': TOL['yellow'],
    'GbgCollect': TOL['cyan'],
    'BuildSnap': TOL['purple'],
    'RecentScan': TOL['green'],
    'HistoryScan': TOL['darkgreen'],
    'Probe': TOL['red'],
    'DeltaScan': TOL['blue'],
}
HATCH_MAP = {
    'RecentScan': '///',
    'HistoryScan': '\\' * 3,
    'Probe': '///',
    'DeltaScan': '\\' * 3,
}
TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
REPAIR_ORDER = {
    'naive': [''],
    'ivmh': [''],
    'heap': ['No Repair', 'Read Repair', 'Write Repair'],
    'chain': ['Write Repair'],
    'par': ['No Repair', 'Read Repair', 'Write Repair'],
}
REPAIR_LABEL = {'': '', 'No Repair': 'NR', 'Read Repair': 'RR', 'Write Repair': 'WR'}
CASE_MAP = {
    'history0': 0.00,
    'history1': 0.01,
}

RUN_STAMP = current_run_stamp()
RUN_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"tc{CONFIG['txn_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"pr{str(CONFIG['probe_ratio']).replace('.', 'p')}",
    f"gc{str(CONFIG['txn_gc_ratio']).replace('.', 'p')}",
    f"re{CONFIG['readable_every']}",
    f"rep{CONFIG['repeat']}",
    f"seed{CONFIG['trace_seed']}",
    RUN_STAMP,
])

BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
    '--seed', str(CONFIG['trace_seed']),
]

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('TAG    :', RUN_TAG)


In [ ]:
def merge_args(base, extra):
    return [*base, *extra]


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    if table_type in BASELINES:
        collapsed = df.groupby(['table_type', 'tx_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'tx_type', 'duration_ms']]
    return df


def aggregate_per_tx(df):
    tx_counts = (
        df.groupby(['table_type', 'repair_type'], as_index=False)
        .size()
        .rename(columns={'size': 'total_tx_count'})
    )
    by_type = (
        df.groupby(['table_type', 'repair_type', 'tx_type'], as_index=False)['duration_ms']
        .sum()
        .merge(tx_counts, on=['table_type', 'repair_type'], how='left')
    )
    by_type['duration_ms'] = by_type['duration_ms'] / by_type['total_tx_count']
    return by_type[['table_type', 'repair_type', 'tx_type', 'duration_ms']]


def scaled_tx_counts(update_share, probe_share, scan_share, delta_share):
    analytical_and_update = 1.0 - CONFIG['txn_gc_ratio']
    update_n = round(CONFIG['txn_count'] * analytical_and_update * update_share)
    probe_n = round(CONFIG['txn_count'] * analytical_and_update * probe_share)
    delta_n = round(CONFIG['txn_count'] * analytical_and_update * delta_share)
    gc_n = round(CONFIG['txn_count'] * CONFIG['txn_gc_ratio'])
    scan_n = CONFIG['txn_count'] - update_n - probe_n - delta_n - gc_n
    return update_n, probe_n, scan_n, delta_n, gc_n


def build_base_sequence():
    update_n, probe_n, scan_n, delta_n, gc_n = scaled_tx_counts(0.20, 0.40, 0.40, 0.0)
    assert delta_n == 0
    seq = ['U'] * update_n + ['P'] * probe_n + ['R'] * scan_n + ['G'] * gc_n
    rng = random.Random(CONFIG['trace_seed'])
    rng.shuffle(seq)
    return seq


BASE_SEQUENCE = build_base_sequence()


def rewrite_history_sequence(history_pct):
    total_history_ops = int(round(CONFIG['txn_count'] * history_pct))
    seq = BASE_SEQUENCE.copy()
    if total_history_ops <= 1:
        history_scan_count = total_history_ops
        history_probe_count = 0
    else:
        history_probe_count = total_history_ops // 2
        history_scan_count = total_history_ops - history_probe_count
    scan_slots = [i for i, op in enumerate(seq) if op == 'R']
    probe_slots = [i for i, op in enumerate(seq) if op == 'P']
    for idx in scan_slots[:history_scan_count]:
        seq[idx] = 'S'
    for idx in probe_slots[:history_probe_count]:
        seq[idx] = 'H'
    return seq


def mix_from_sequence(seq):
    update_n = sum(op == 'U' for op in seq)
    probe_n = sum(op in {'P', 'H'} for op in seq)
    scan_n = sum(op in {'R', 'S'} for op in seq)
    delta_n = sum(op == 'D' for op in seq)
    non_gc = update_n + probe_n + scan_n + delta_n
    return {
        'update_ratio': update_n / non_gc,
        'probe_ratio': probe_n / non_gc,
        'scan_ratio': scan_n / non_gc,
        'delta_ratio': delta_n / non_gc,
    }


def build_manual_args(seq):
    mix = mix_from_sequence(seq)
    args = [
        '--txn-update-ratio', str(mix['update_ratio']),
        '--txn-probe-ratio', str(mix['probe_ratio']),
        '--txn-scan-ratio', str(mix['scan_ratio']),
        '--txn-delta-ratio', str(mix['delta_ratio']),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', '0.0',
        '--probe-history-ratio', '0.0',
        '--manual-txs', ''.join(seq),
    ]
    if any(op in {'H', 'S'} for op in seq):
        args.append('--distinct-history-targets')
    return args


def run_case(case_name, history_pct):
    csv_path = DATA_DIR / f'{case_name}_{RUN_TAG}.csv'
    if CONFIG['force_rerun'] and csv_path.exists():
        csv_path.unlink()
    if csv_path.exists():
        return pd.read_csv(csv_path, keep_default_na=False), rewrite_history_sequence(history_pct)

    seq = rewrite_history_sequence(history_pct)
    args = build_manual_args(seq)
    rows = []
    for table_type in TABLE_TYPES:
        print(f'  case={case_name} table={table_type}')
        trials = []
        for trial in range(CONFIG['repeat']):
            result = run_checked([str(BIN), *merge_args(BASE_ARGS, args), '--table-type', table_type], ROOT, quiet=True, timeout=CONFIG['timeout_sec'])
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {case_name}/{table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            df['trial'] = trial
            agg = aggregate_per_tx(df)
            agg['trial'] = trial
            trials.append(agg)
        df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
        df_avg = (
            df_all.groupby(['table_type', 'repair_type', 'tx_type'], as_index=False)['duration_ms']
            .mean()
        )
        rows.append(collapse_repairs(df_avg, table_type))

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print('Saved', csv_path)
    return out, seq


def seq_counts(seq):
    return {ch: seq.count(ch) for ch in ['U', 'P', 'H', 'R', 'S', 'D', 'G']}


def changed_positions(base, other):
    return [(i, a, b) for i, (a, b) in enumerate(zip(base, other)) if a != b]


In [ ]:
results = {}
sequences = {}
for case_name, history_pct in CASE_MAP.items():
    df, seq = run_case(case_name, history_pct)
    results[case_name] = df
    sequences[case_name] = seq

base_counts = seq_counts(BASE_SEQUENCE)
case_counts = {name: seq_counts(seq) for name, seq in sequences.items()}
changes_01 = changed_positions(sequences['history0'], sequences['history1'])

print('Base counts:', base_counts)
print('History 0 counts:', case_counts['history0'])
print('History 1 counts:', case_counts['history1'])
print('Changed positions from 0% to 1%:', changes_01[:10])

summary = []
for case_name, df in results.items():
    total = df.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].sum()
    total['case'] = case_name
    summary.append(total)
display(pd.concat(summary, ignore_index=True).sort_values(['case', 'table_type', 'repair_type']).reset_index(drop=True))


In [ ]:
def build_legend_handles():
    legend_handles = []
    legend_labels = []
    for tx in TX_ORDER[::-1]:
        if tx in WRITE_OPS:
            patch = Patch(facecolor=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
        else:
            patch = Patch(facecolor='white', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
        legend_handles.append(patch)
        legend_labels.append(tx)
    return legend_handles, legend_labels


def panel_total_max(df):
    pivot = df.pivot_table(index=['table_type', 'repair_type'], columns='tx_type', values='duration_ms', aggfunc='sum', fill_value=0.0)
    pivot = pivot.reindex(columns=TX_ORDER, fill_value=0.0)
    if pivot.empty:
        return 0.0
    return float(pivot.sum(axis=1).max())


def plot_panel(ax, df, y_max):
    pivot = df.pivot_table(index=['table_type', 'repair_type'], columns='tx_type', values='duration_ms', aggfunc='sum', fill_value=0.0)
    pivot = pivot.reindex(columns=TX_ORDER, fill_value=0.0)

    bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
    x = 0.0
    gap = 0.36
    for table in TABLE_ORDER:
        subs = REPAIR_ORDER[table]
        start = x
        for repair in subs:
            combos.append((table, repair))
            bar_x.append(x)
            minor_labels.append(REPAIR_LABEL[repair])
            x += 0.78
        major_centers.append((start + (x - 0.78)) / 2.0)
        major_labels.append(display_name(table, ''))
        x += gap

    for idx, (table, repair) in enumerate(combos):
        if (table, repair) in pivot.index:
            row = pivot.loc[(table, repair)]
        else:
            row = pd.Series(0.0, index=TX_ORDER)
        bottom = 0.0
        for tx in TX_ORDER:
            value = float(row.get(tx, 0.0))
            if value <= 0:
                continue
            if tx in WRITE_OPS:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
            else:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='white', edgecolor='black', linewidth=0.4)
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='none', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
            bottom += value

    ax.set_xticks([])
    ax.set_ylabel('Duration (ms / tx)')
    ax.set_ylim(0, y_max)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    for xi, label in zip(bar_x, minor_labels):
        ax.text(xi, -0.06, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=8, clip_on=False)
    for xc, label in zip(major_centers, major_labels):
        ax.text(xc, -0.14, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=9, clip_on=False)


legend_handles, legend_labels = build_legend_handles()
y_max = max(panel_total_max(df) for df in results.values()) * 1.08

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6), sharey=True)
plot_panel(axes[0], results['history0'], y_max)
plot_panel(axes[1], results['history1'], y_max)
axes[0].set_title('History 0%')
axes[1].set_title('History 1%')
axes[1].legend(legend_handles, legend_labels, loc='upper right', ncol=2, framealpha=0.95)
fig.tight_layout(rect=[0, 0, 1, 1])
combined_pdf = FIGS_DIR / f'exp2_wc2_history0_vs_1_breakdown_{RUN_TAG}.pdf'
latest_combined_pdf = FIGS_DIR / 'exp2-wc2-history0-vs-1-breakdown.pdf'
fig.savefig(combined_pdf, format='pdf', bbox_inches='tight')
fig.savefig(latest_combined_pdf, format='pdf', bbox_inches='tight')
plt.show()
print('Saved', combined_pdf)
print('Saved', latest_combined_pdf)

single_names = {
    'history0': 'exp2-wc2-history0-breakdown.pdf',
    'history1': 'exp2-wc2-history1-breakdown.pdf',
}
for case_name, title in [('history0', 'History 0%'), ('history1', 'History 1%')]:
    fig, ax = plt.subplots(1, 1, figsize=(4.8, 4.4))
    plot_panel(ax, results[case_name], y_max)
    ax.set_title(title)
    ax.legend(legend_handles, legend_labels, loc='upper right', ncol=2, framealpha=0.95)
    fig.tight_layout(rect=[0, 0, 1, 1])
    out_pdf = FIGS_DIR / f'{case_name}_{RUN_TAG}.pdf'
    latest_pdf = FIGS_DIR / single_names[case_name]
    fig.savefig(out_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
    plt.show()
    print('Saved', out_pdf)
    print('Saved', latest_pdf)
